1.  Import essential tools and set up OpenAI's API environment

In [1]:
# Run this in your Colab notebook
!pip install langchain langchain-openai openai langchain-community PyPDF faiss-cpu chromadb tiktoken langchain-chroma


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.2/331.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4

In [2]:
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

In [3]:
from langchain_openai import ChatOpenAI
model_op = ChatOpenAI(model = 'gpt-4', openai_api_key=api_key)

2. Load Nestle's HR policy using PyPDFLoader and split it for easy processing

In [4]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader('/content/1728286846_the_nestle_hr_policy_pdf_2012.pdf')
docs_pdf = loader.load()
print(len(docs_pdf))
print(docs_pdf[0].page_content)
print(docs_pdf[0].metadata)

8
Policy
Mandatory
September   2012
The Nestlé  
Human Resources Policy
{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2013-10-31T10:20:17+01:00', 'trapped': '/False', 'source': '/content/1728286846_the_nestle_hr_policy_pdf_2012.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
   chunk_size=300,
   chunk_overlap=80,
)

chunk = splitter.split_documents(docs)

print("Number of Chunks",len(chunk))
print("Sample Text", chunk[1].page_content[:400])
print("Sample Metadata", chunk[1].metadata)


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size= 250,
    chunk_overlap=50
)

split_doc = text_splitter.split_documents(docs_pdf)

print(f"Number of chunks : {len(split_doc)}")
for i, doc in enumerate(split_doc[:8]):
    print(doc.metadata)
    print(doc.page_content)

Number of chunks : 77
{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2013-10-31T10:20:17+01:00', 'trapped': '/False', 'source': '/content/1728286846_the_nestle_hr_policy_pdf_2012.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}
Policy
Mandatory
September   2012
The Nestlé  
Human Resources Policy
{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2013-10-31T10:20:17+01:00', 'trapped': '/False', 'source': '/content/1728286846_the_nestle_hr_policy_pdf_2012.pdf', 'total_pages': 8, 'page': 1, 'page_label': '2'}
Policy
Mandatory
September 
 20
12
Issuing  departement
Hum
an Resources
T arget  audience  
All
 employees
Approver
Executive Board, Nestlé S.A.
Repository
All Nestlé Principles and Policies, Standards and
{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', '

3.  Create vector representations for text chunks using Chroma dB and OpenAI's embeddings.

In [23]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [29]:
from langchain_core.documents import Document

In [12]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model = "text-embedding-3-small",openai_api_key=api_key)

texts = [doc.page_content for doc in split_doc]

vectors = embedding_model.embed_documents(texts)

In [33]:
vector_store_chroma = Chroma(
    embedding_function=OpenAIEmbeddings(model = "text-embedding-3-small",openai_api_key=api_key),
    persist_directory='my_chroma_db_q',
    collection_name='sample'
)

In [35]:
vector_store_chroma.add_documents(docs_pdf) ##/content/1728286846_the_nestle_hr_policy_pdf_2012.pdf

['335d918c-5b43-493c-9a93-e83f5a8965b3',
 'a8100515-7db5-4453-b864-ba3446fc66ab',
 '2062736b-05f4-4357-ab66-3de925d270b4',
 '0bb3f236-270c-4416-b9e0-fdb265a8a77c',
 '23186f18-e016-43d0-a7ac-34c9bfba88da',
 'fd54eba8-a4fa-4204-a369-8e09eb92bf5a',
 '1388ec3a-5815-42c7-b152-f25d8224a9f6',
 '4f2afb28-5fda-4154-b61c-0375714cc8d5']

In [36]:
docs_pdf

[Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2013-10-31T10:20:17+01:00', 'trapped': '/False', 'source': '/content/1728286846_the_nestle_hr_policy_pdf_2012.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='Policy\nMandatory\nSeptember\u2009 \u20092012\nThe Nestlé  \nHuman Resources Policy'),
 Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2013-10-31T10:20:17+01:00', 'trapped': '/False', 'source': '/content/1728286846_the_nestle_hr_policy_pdf_2012.pdf', 'total_pages': 8, 'page': 1, 'page_label': '2'}, page_content='Policy\nMandatory\nSeptember\u2009\n\u200920\n12\nIssuing \u2009departement\nHum\nan Resources\nT arget \u2009audience \u2009\nAll\n employees\nApprover\nExecutive Board, Nestlé S.A.\nRepository\nAll Nestlé Principles and Policies,

In [37]:
vector_store_chroma.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['335d918c-5b43-493c-9a93-e83f5a8965b3',
  'a8100515-7db5-4453-b864-ba3446fc66ab',
  '2062736b-05f4-4357-ab66-3de925d270b4',
  '0bb3f236-270c-4416-b9e0-fdb265a8a77c',
  '23186f18-e016-43d0-a7ac-34c9bfba88da',
  'fd54eba8-a4fa-4204-a369-8e09eb92bf5a',
  '1388ec3a-5815-42c7-b152-f25d8224a9f6',
  '4f2afb28-5fda-4154-b61c-0375714cc8d5'],
 'embeddings': array([[ 0.00010793,  0.03456473,  0.03349791, ...,  0.01741572,
          0.02516344, -0.03179102],
        [ 0.00938267,  0.03824986,  0.04496223, ...,  0.00574014,
          0.03705122, -0.0296463 ],
        [ 0.00156692,  0.03605103,  0.02336021, ...,  0.00602844,
          0.01283436, -0.02605148],
        ...,
        [ 0.00446345,  0.03352658,  0.06081448, ..., -0.01715638,
          0.00947215, -0.03738138],
        [ 0.00733594,  0.02543211,  0.01705734, ..., -0.01310722,
          0.01524901, -0.01414605],
        [ 0.01536839, -0.0348107 , -0.00932883, ..., -0.01359298,
          0.01769069,  0.01902224]]),
 'documents': [

In [38]:
vector_store_chroma.add_documents(split_doc)

['915950a3-df26-4e53-9508-2fd38284faa2',
 '13373a5e-e5f7-47cd-88d9-c5a5a7faebc3',
 '9e5270ae-9ce8-449c-b15c-0c82a8295f90',
 '7262a581-208e-4c09-a747-4a389bc35f3d',
 'e8f6ed70-ab09-4809-9648-00414d3d0984',
 '8884f7c9-6aae-4e9f-b7f5-2e976f89c557',
 '11f16066-8fb0-49ef-b9f4-96541d1c742b',
 'a4ac9971-8b8d-4e7f-bff4-c80528e1d2d2',
 'fe9789ca-91d9-469c-9e88-af078f7c96c5',
 '24ba03bf-5538-47fa-81b6-911ef66507a8',
 '4fbac4b2-dfba-4b76-9dd8-a53fc3b1e8fc',
 'c78b71e1-dbd8-4381-b752-1e2497857ccb',
 '9c9185b9-9637-490b-81a8-2211937ec748',
 '680401dc-08af-47a8-9910-11ea0976a665',
 '28ef391e-60d5-427a-b55c-d58ce9e5f8be',
 '5bf63b3b-accc-4b2d-98d5-c97254bee791',
 'dd79429c-e034-48e3-932d-0e12fcc29766',
 'e27ca83e-72ac-47ec-9dc0-15ba85e82ec9',
 '897e961e-83e2-446b-aaf0-e04e3dea21a8',
 '6d22c163-b553-4391-812a-e0f1c57e4f48',
 '8be77a50-84db-4d26-9634-e907075e5c10',
 '9b48c239-c0cd-4325-8e18-f5e930098705',
 '351d0ebd-5c7c-44a2-8c2a-e41bd3130110',
 '5543a352-a692-4673-99e5-278f138b456d',
 'a873d8ea-331d-

In [39]:
retriever = vector_store_chroma.as_retriever(search_kwargs={"k": 3})

In [40]:
query = "What does Nestle do for their employees?"
results = retriever.invoke(query)

In [41]:
for i, doc in enumerate(results):
   print(f"\n--- Result {i+1} ---")
   print(doc.page_content)



--- Result 1 ---
The Nestlé Human Resources Policy
3
 Total rewards
Attracting new hires and keeping current 
employees engaged is not only about 
remuneration and benefits based on solid 
performance. It is also about the hard earned 
value and trust that our name brings to those 
who work with us; the relationships with our line 
managers and fellow workers; recognition and 
experiences enjoyed while working for a diverse 
global company; and possibilities to learn and 
grow. These are as a whole, the Total Rewards we 
receive.
Nestlé, therefore, focuses on Fixed Pay, 
Variable Pay, Benefits, Personal Growth and 
Development and Work Life Environment as the 
key elements that define Total Rewards. In the 
spirit of developing a high performance culture, 
those elements need to correspond to what is 
valued by employees in each and every market, 
and which demonstrate how Nestlé is committed 
to giving each employee the opportunity to grow, 
evolve and contribute.
Nestlé Total Reward

In [42]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma


from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


In [44]:
nestle_pdf_path = '/content/1728286846_the_nestle_hr_policy_pdf_2012.pdf'
loader = PyPDFLoader(nestle_pdf_path)
docs = loader.load()

print("Pages Loaded",len(docs))
print("Sample Text", docs[1].page_content[:400])
print("Sample Metadata", docs[1].metadata)


Pages Loaded 8
Sample Text Policy
Mandatory
September 
 20
12
Issuing  departement
Hum
an Resources
T arget  audience  
All
 employees
Approver
Executive Board, Nestlé S.A.
Repository
All Nestlé Principles and Policies, Standards and  
Guidelines can be found in the Centre online repository at:  
http://intranet.nestle.com/nestledocs
Copyright
 and  confidentiality
Al
l rights belong to Nestec Ltd., Vevey, Switzerland.
© 20
Sample Metadata {'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2013-10-31T10:20:17+01:00', 'trapped': '/False', 'source': '/content/1728286846_the_nestle_hr_policy_pdf_2012.pdf', 'total_pages': 8, 'page': 1, 'page_label': '2'}


In [48]:
text_splitter = RecursiveCharacterTextSplitter(
   chunk_size=300,
   chunk_overlap=80,
)

chunk = text_splitter.split_documents(docs)

print("Number of Chunks",len(chunk))
print("Sample Text", chunk[1].page_content[:400])
print("Sample Metadata", chunk[1].metadata)

Number of Chunks 66
Sample Text Policy
Mandatory
September 
 20
12
Issuing  departement
Hum
an Resources
T arget  audience  
All
 employees
Approver
Executive Board, Nestlé S.A.
Repository
All Nestlé Principles and Policies, Standards and  
Guidelines can be found in the Centre online repository at:
Sample Metadata {'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2013-10-31T10:20:17+01:00', 'trapped': '/False', 'source': '/content/1728286846_the_nestle_hr_policy_pdf_2012.pdf', 'total_pages': 8, 'page': 1, 'page_label': '2'}


In [49]:
embedding = OpenAIEmbeddings(model = "text-embedding-3-small", openai_api_key = api_key)
vectorstore_FAISS = FAISS.from_documents(chunk, embedding)


In [50]:
retriever = vectorstore_FAISS.as_retriever(search_kwargs={"k": 3})

In [51]:
def format_docs(docs):
   return "\n\n".join(
       f"[page = {d.metadata.get("page","?")}] {d.page_content}"
       for d in docs
       )


In [52]:
nestle_prompt = ChatPromptTemplate.from_messages([
   ("system",
    "You are very helpful. You Answer ONLY using the provided context."
    "You draft an E-Mail which contain the answer and the E-Mail should not more than 250 words. It should be very sharp and concise."
    "If the answer is not in the context, SAY: 'I don't know based on the provided context."),
   ("human",
    "Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer:")
])


In [54]:
Nestle_RAG = ChatOpenAI(model="gpt-4", openai_api_key=api_key)

parser = StrOutputParser()


In [55]:
rag_chain = (
   {
       "context": retriever | format_docs,
       "question": RunnablePassthrough()
   }
   | nestle_prompt
   | Nestle_RAG
   | parser
)


In [56]:
query = "What does Nestle do for their employees?"

answer = rag_chain.invoke(query)

print(answer)


Subject: Nestlé's Commitment towards Employee Engagement 

Dear [Recipient's Name],

Nestlé's well-structured Human Resources Policy ensures employee satisfaction and overall corporate growth. At Nestlé, attracting new hires and retaining existing employees is not only focused on remuneration and benefits rooted in stellar performance. 

The organization values the trust that its name brings to all stakeholders, strive to instill a sense of personal commitment in the workforce, and encourages its line managers to build and sustain a conducive work environment conducive to success.

In addition, Nestlé upholds a culture built on trust, mutual respect, and dialogue since its inception. Its global management and workforce constantly work together to foster positive individual and collective relationships.

Hence, Nestlé's dedication to its employees is quite evident in its comprehensive human resources policy.

Kind regards,

[Your Name]


4.  Build a question-answering system using the GPT-3.5 Turbo model to retrieve answers from text chunks.

In [ ]:
#!pip install gradio
#import gradio as gr

In [ ]:
#def greet(name):
    return "Hello you are welcome to Gradio" + name + "!"

demo = gr.Interface(fn=greet, inputs="text", outputs="text")
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d145e8c73dcf697e17.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

In [ ]:
from langchain_openai import ChatOpenAI
model_op = ChatOpenAI(model = 'gpt-4', openai_api_key=api_key)

In [ ]:
import gradio as gr

def greet(name, intensity):
    return "Hello, " + name + "!" * int(intensity)

demo = gr.Interface(
    fn=greet,
    inputs=["text", "slider"],
    outputs=["text"],
    api_name="predict"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://02170fbf0160ae0e4e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
from langchain.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI

model_op = ChatOpenAI(model = 'gpt-4', openai_api_key=api_key)


def predict(message, history):
    history_langchain_format = []
    for msg in history:
        if msg["role"] == "user":
            history_langchain_format.append(HumanMessage(content=msg["content"]))
        elif msg["role"] == "assistant":
            history_langchain_format.append(AIMessage(content=msg["content"]))
    history_langchain_format.append(HumanMessage(content=message))
    gpt_response = model.invoke(history_langchain_format)
    return gpt_response.content


demo = gr.ChatInterface(
    predict,
    api_name= 'openai_api_key=api_key',
)

demo.launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57c5ee7e44e5910dd5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
